# Tool使用的概述

## 1.工具的调用方式

### 1.1 方式一：直接调用

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    参数：
        city:城市名称，如“北京”

    返回：
        天气信息字符串
    """
    return city + "晴天"

In [ ]:
get_weather.invoke({"city":"北京"})

### 1.2 基于模型进行调用

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)


In [ ]:
from langchain_core.tools import tool

# 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return "晴天"

# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# AI 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")
# response = model_with_tools.invoke("2 + 3 = ？")

# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

## 2.从Message流转看工具的调用

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.messages import HumanMessage, ToolMessage
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

# 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return "晴天"

# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# 声明一个消息列表
messages = [
    HumanMessage("今天北京天气如何?")
]

# 添加AIMessage到消息列表中
messages.append(response)

# rprint(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    # 返回的是ToolMessage的类型消息
    tool_res = get_weather.invoke(tool_call)
    rprint(type(tool_res))
    messages.append(tool_res)

rprint("==========================> messages <===================")
for msg in messages:
    msg.pretty_print()
rprint("==========================> messages <===================")
final_res = model_with_tools.invoke(messages)
rprint(f"final_res:\n{final_res}")